# 🗣️ VoxCPM2 (2B) — Batch Voice Cloning Test

Runs **12 test texts** through VoxCPM2 in a single execution.
Saves all outputs to `outputs/voxcpm2/` and zips for download.

## ▶️ Instructions
1. **Runtime → Change runtime type → GPU** (T4 is plenty)
2. Run all cells top to bottom
3. Upload your reference voice when prompted
4. Wait for all 12 texts to generate
5. Download the zip at the end

## 1. Setup and Installation

In [ ]:
import sys, subprocess, shutil

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("voxcpm")
pip_install("soundfile", "librosa")

if shutil.which("ffmpeg") is None:
    print("⚠️ ffmpeg not found. WAV will still work, but MP3/M4A may not.")
else:
    print("✅ ffmpeg found:", shutil.which("ffmpeg"))
print("✅ Install step finished.")

## 2. Imports & Device

In [ ]:
import os, gc, warnings, time, json
import numpy as np
import torch

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Apple MPS device. Synthesis will be slower than GPU.")
else:
    DEVICE = "cpu"
    print("⚠️ No GPU/MPS detected — running on CPU. This will be SLOW.")

print("   torch:", torch.__version__, "| device:", DEVICE)

## 3. Configuration + Test Texts

In [ ]:
# --- Model -------------------------------------------------------------------
MODEL_ID      = "openbmb/VoxCPM2"
LOAD_DENOISER = False

# --- Cloning mode ------------------------------------------------------------
CLONE_MODE    = "ultimate"

# --- Reference voice ---------------------------------------------------------
REF_AUDIO_PATH = ""
PROMPT_TEXT    = "Replace this text with the exact words spoken in your reference audio sample."

# --- Generation knobs --------------------------------------------------------
CFG_VALUE           = 2.0
INFERENCE_TIMESTEPS = 10
DENOISE             = False
NORMALIZE           = False

# --- Audio preprocessing -----------------------------------------------------
REF_TARGET_SR  = 16000
TRIM_TOP_DB    = 30
PEAK_NORM      = 0.95
IDEAL_MIN_SEC  = 5
IDEAL_MAX_SEC  = 30

# --- 12 Test Texts -----------------------------------------------------------
TEST_TEXTS = [
    {"id": 1, "text": "Your ticket number is B 4 7 2 9 and the fare is rupees three thousand two hundred.", "language": "English", "category": "Booking Confirmation"},
    {"id": 2, "text": "Thank you for calling customer support. Your query has been registered and our team will get back to you within twenty four hours. We apologise for the inconvenience caused.", "language": "English", "category": "Customer Support"},
    {"id": 3, "text": "Departure at 06:45 AM on 3rd February 2025", "language": "English", "category": "Flight Details"},
    {"id": 4, "text": "Aapki booking confirm ho gayi. Reference number note kar lijiye: B 4 9 2 1.", "language": "Hindi", "category": "Booking Confirmation"},
    {"id": 5, "text": "Namaskar aur hamare service mein aapka swagat hai. Aapka loan application approved ho gaya hai. Amount aapke registered account mein do se teen working days mein credit ho jayega. Kisi bhi sahayta ke liye humse contact karein.", "language": "Hindi", "category": "Loan / Finance"},
    {"id": 6, "text": "Flight booking ke liye 1 dabayen. Flight status ke liye 2 dabayen. Cancellation ke liye 3 dabayen.", "language": "Hindi", "category": "IVR Menu"},
    {"id": 7, "text": "Dhanyavaad IndiGo ko call karne ke liye. Aapka din mangalmay ho.", "language": "Hindi", "category": "Call Closing"},
    {"id": 8, "text": "Aapka PNR number hai A B 1 2 3 4. Ise save kar lijiye.", "language": "Hindi", "category": "Booking Confirmation"},
    {"id": 9, "text": "Kya aap travel insurance add karna chahenge? Yeh sirf rupees 299 mein available hai.", "language": "Hindi", "category": "Upsell / Add-on"},
    {"id": 10, "text": "Yeh final boarding call hai passengers Mr. Sharma aur Mrs. Gupta ke liye, flight 6E 888 ke liye gate C 3 par.", "language": "Hindi", "category": "Boarding Announcement"},
    {"id": 11, "text": "IndiGo BluChip Gold members aur business class passengers priority boarding le sakte hain.", "language": "Hindi", "category": "Boarding Announcement"},
    {"id": 12, "text": "IndiGo wallet mein minimum rupees 500 add kar sakte hain future bookings ke liye.", "language": "Hindi", "category": "Wallet / Payment"}
]

OUTPUT_DIR = "outputs/voxcpm2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config loaded. {len(TEST_TEXTS)} test texts ready.")
for t in TEST_TEXTS:
    print(f"  [{t['id']:2d}] [{t['language']:7s}] [{t['category']:22s}] {t['text'][:60]}...")

## 4. Upload / Load Voice Sample

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not REF_AUDIO_PATH:
    if IN_COLAB:
        print("Upload your voice sample (WAV/MP3/M4A/FLAC, ~5-30s):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        REF_AUDIO_PATH = list(uploaded.keys())[0]
    else:
        raise RuntimeError("REF_AUDIO_PATH is empty and not running in Colab.")

if not os.path.exists(REF_AUDIO_PATH):
    raise FileNotFoundError(f"Audio file not found: {REF_AUDIO_PATH}")
print("✅ Reference audio:", REF_AUDIO_PATH)

## 5. Audio Preprocessing

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display

def load_and_preprocess(path):
    """Return a cleaned float32 mono waveform at REF_TARGET_SR."""
    try:
        wav, sr = librosa.load(path, sr=REF_TARGET_SR, mono=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not decode '{path}'. Unsupported format or missing ffmpeg.\n"
            f"Original error: {e}"
        )
    wav, _ = librosa.effects.trim(wav, top_db=TRIM_TOP_DB)
    peak = float(np.max(np.abs(wav))) if wav.size else 0.0
    if peak > 0:
        wav = (wav / peak) * PEAK_NORM
    return wav.astype(np.float32), REF_TARGET_SR

ref_wav, ref_sr = load_and_preprocess(REF_AUDIO_PATH)
duration = len(ref_wav) / ref_sr

CLEAN_REF_PATH = "reference_clean.wav"
sf.write(CLEAN_REF_PATH, ref_wav, ref_sr)
print(f"Processed: {duration:.1f}s, mono, {ref_sr} Hz -> {CLEAN_REF_PATH}")

if duration < 2:
    print(f"🚨 WARNING: only {duration:.1f}s — very short.")
elif duration < IDEAL_MIN_SEC:
    print(f"⚠️ {duration:.1f}s is a little short; {IDEAL_MIN_SEC}-{IDEAL_MAX_SEC}s is ideal.")
elif duration > IDEAL_MAX_SEC:
    print(f"⚠️ {duration:.1f}s is longer than {IDEAL_MAX_SEC}s; fine, but not necessary.")
else:
    print("✅ Duration is in a good range for cloning.")

print("Preview of your processed reference:")
display(Audio(ref_wav, rate=ref_sr))

## 6. Model Loading

In [ ]:
from voxcpm import VoxCPM

try:
    model = VoxCPM.from_pretrained(
        MODEL_ID,
        load_denoiser=LOAD_DENOISER,
        device=DEVICE,
    )
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); gc.collect()
    raise RuntimeError("CUDA OOM while loading VoxCPM2.")
except Exception as e:
    raise RuntimeError(f"Model download/load failed: {e}")

SAMPLE_RATE = model.tts_model.sample_rate
print(f"✅ VoxCPM2 ready. Output sample rate: {SAMPLE_RATE} Hz")

## 7. 🔄 Batch Generation — All 12 Test Texts

This cell loops through all 12 test texts, generates audio for each,
and saves the outputs. **No need to re-run anything manually!**

In [ ]:
results = []

print(f"\n{'='*70}")
print(f"  BATCH GENERATION: {len(TEST_TEXTS)} texts through VoxCPM2")
print(f"{'='*70}\n")

for idx, test in enumerate(TEST_TEXTS):
    test_id = test["id"]
    text = test["text"]
    wav_filename = f"test_{test_id:02d}.wav"
    wav_path = os.path.join(OUTPUT_DIR, wav_filename)

    print(f"\n--- [{idx+1}/{len(TEST_TEXTS)}] Test #{test_id}: {test['category']} ({test['language']}) ---")
    print(f"    Text: {text[:80]}{'...' if len(text)>80 else ''}")

    start_time = time.time()
    status = "success"
    error_msg = None
    audio_duration = 0

    try:
        gen_kwargs = dict(
            text=text,
            reference_wav_path=CLEAN_REF_PATH,
            cfg_value=CFG_VALUE,
            inference_timesteps=INFERENCE_TIMESTEPS,
            normalize=NORMALIZE,
            denoise=DENOISE,
        )

        if CLONE_MODE == "ultimate":
            gen_kwargs["prompt_wav_path"] = CLEAN_REF_PATH
            gen_kwargs["prompt_text"] = PROMPT_TEXT

        audio_np = model.generate(**gen_kwargs)
        audio_np = np.asarray(audio_np, dtype=np.float32).squeeze()

        sf.write(wav_path, audio_np, SAMPLE_RATE)
        gen_time = time.time() - start_time
        audio_duration = len(audio_np) / SAMPLE_RATE

        print(f"    ✅ Saved: {wav_filename} ({audio_duration:.1f}s audio, generated in {gen_time:.1f}s)")
        display(Audio(audio_np, rate=SAMPLE_RATE))

    except Exception as e:
        gen_time = time.time() - start_time
        status = "error"
        error_msg = str(e)
        print(f"    ❌ Error: {e}")
        torch.cuda.empty_cache()
        gc.collect()

    results.append({
        "id": test_id,
        "text": text,
        "language": test["language"],
        "category": test["category"],
        "wav_file": wav_filename,
        "generation_time_sec": round(gen_time, 2),
        "audio_duration_sec": round(audio_duration, 2),
        "status": status,
        "error": error_msg
    })

# Save metadata
metadata = {"model": "voxcpm2", "model_name": MODEL_ID, "results": results}
meta_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n{'='*70}")
print(f"  BATCH COMPLETE: {sum(1 for r in results if r['status']=='success')}/{len(results)} successful")
print(f"  Metadata saved: {meta_path}")
print(f"{'='*70}")

## 8. Download Results

In [ ]:
import shutil

zip_name = "voxcpm2_outputs"
shutil.make_archive(zip_name, "zip", "outputs/voxcpm2")
print(f"✅ Created {zip_name}.zip")

# Summary table
print(f"\n{'ID':>3} | {'Status':>7} | {'Time':>6} | {'Duration':>8} | {'Language':>8} | {'Category':<22} | Text")
print("-" * 110)
for r in results:
    print(f"{r['id']:3d} | {r['status']:>7} | {r['generation_time_sec']:5.1f}s | {r['audio_duration_sec']:6.1f}s | {r['language']:>8} | {r['category']:<22} | {r['text'][:40]}...")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
except:
    print(f"\nDownload {zip_name}.zip from the file browser.")